In [1]:
!pip install -q transformers==4.44.2 sentencepiece x-transformers optuna scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.3/120.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.

# WangchanBERTa + x-transformers: Strong Optimized Version

เวอร์ชันนี้ปรับจากโค้ดเดิมเพื่อให้ optimized มีโอกาสชนะ non-optimized มากขึ้น โดยแก้จุดที่ทำให้ตัว optimized เดิมเสีย precision และ F1

แนวทางที่เพิ่มเข้าไป:

- ใช้ dataset ใหม่ตัวเดียว `SIED_Thai_FINAL_20000_balanced_multilabel_adjusted.csv`
- Split เป็น train / validation / test ภายในไฟล์เดียว
- ใส่ baseline/non-optimized configuration เป็น Optuna trial แรก เพื่อไม่ให้ search แย่กว่า baseline ง่าย ๆ
- ปรับ search space ให้แคบลงและเหมาะกับ dataset นี้มากขึ้น
- Tune `pos_weight_scale` เพื่อแก้ปัญหา recall สูงแต่ precision ตก
- Tune threshold แบบละเอียดกว่าเดิม
- Objective ใหม่เน้น `macro_f1 + micro_f1 + min_per_label_f1 + subset_accuracy`
- รายงานผลครบ: train / validation / test, per-label report, confusion matrix, predicted count, threshold
- Encoder ยังคง frozen เพื่อรักษาแนวทาง fixed encoder + tuned decoder


In [2]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    hamming_loss,
    multilabel_confusion_matrix
)

import optuna


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

DEVICE: cuda


In [5]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128
EPOCHS = 100
BATCH_SIZE = 64
NUM_LABELS = 6

# เพิ่มจำนวน trial จากเดิม เพื่อให้ optimization มีโอกาสชนะ baseline มากขึ้น
# ถ้า Colab เวลาน้อย ใช้ 15 ได้ แต่ถ้าต้องการผลดีที่สุด แนะนำ 25-30
N_TRIALS = 25

# ใช้ epoch สั้นกว่าระหว่าง search เพื่อประหยัดเวลา
# final model ยัง train 100 epochs ตามเดิม
OPTUNA_EPOCHS = 35

LABEL_COLUMNS = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear"
]

TEXT_COLUMN = "Tweets"

CSV_PATH = "/kaggle/input/datasets/kmkimmy/sied-thai-final-20000-balanced-multilabel-adjusted/SIED_Thai_FINAL_20000_balanced_multilabel_adjusted.csv"

if not os.path.exists(CSV_PATH):
    CSV_PATH = "/mnt/data/SIED_Thai_FINAL_20000_balanced_multilabel_adjusted.csv"

# ค่า non-optimized ที่ผู้ใช้รายงานไว้ ใช้เป็น reference ตอนอ่านผล
NON_OPTIMIZED_REFERENCE = {
    "macro_f1_estimated_from_per_label": 0.8234,
    "happiness_f1": 0.8398,
    "sadness_f1": 0.8334,
    "anger_f1": 0.8131,
    "disgust_f1": 0.7892,
    "surprise_f1": 0.8310,
    "fear_f1": 0.8341
}


In [6]:
# ===== CHECKPOINT / RESUME CONFIG (Kaggle) =====
# กัน session Kaggle หลุดระหว่าง final training 100 epochs
#
# วิธีทำงานบน Kaggle:
#   - WRITE: เขียน checkpoint ลง /kaggle/working (เขียนได้ที่เดียว)
#     * ภายใน session เดียว: rerun ได้ resume ต่อทันที
#     * ข้าม session: ต้อง "Save Version" (commit) เพื่อให้ /kaggle/working ถูกเก็บ
#   - READ (resume ข้าม session): เอา output ที่เซฟไว้ไปสร้าง/แนบเป็น Kaggle Dataset
#     แล้วตั้ง RESUME_INPUT_DIR ให้ชี้ไป /kaggle/input/<dataset> (read-only)
#     (เป็น pattern เดียวกับที่ใช้ในโน้ตบุ๊ก mbert: โหลด checkpoint จาก /kaggle/input/...)

import json

# WRITE target — เขียนได้เฉพาะ /kaggle/working
CKPT_DIR = "/kaggle/working/sied_strong_optimized"
os.makedirs(CKPT_DIR, exist_ok=True)

# READ fallback สำหรับ resume ข้าม session — ตั้งเป็น path ของ dataset ที่ attach ไว้
# ถ้าเทรนต่อเนื่องใน session เดียว ปล่อยเป็น None ได้เลย
RESUME_INPUT_DIR = "/kaggle/input/datasets/kmkimmy/sied-strong-optimized"   # ตัวอย่าง: "/kaggle/input/sied-strong-optimized-ckpt"

BEST_PARAMS_PATH = os.path.join(CKPT_DIR, "best_params.json")
FINAL_CKPT_PATH  = os.path.join(CKPT_DIR, "final_checkpoint.pt")
CHECKPOINT_EVERY = 5

def resolve_resume(filename, write_path):
    """หา path ที่จะอ่าน checkpoint/best_params: ดู /kaggle/working ก่อน แล้วค่อย input dataset"""
    if os.path.exists(write_path):
        return write_path
    if RESUME_INPUT_DIR:
        cand = os.path.join(RESUME_INPUT_DIR, filename)
        if os.path.exists(cand):
            return cand
    return None

print("CKPT_DIR        :", CKPT_DIR)
print("RESUME_INPUT_DIR:", RESUME_INPUT_DIR)
print("BEST_PARAMS_PATH:", BEST_PARAMS_PATH)
print("FINAL_CKPT_PATH :", FINAL_CKPT_PATH)
print("⚠️  อย่าลืม 'Save Version' (commit) เพื่อให้ checkpoint ใน /kaggle/working ไม่หายเมื่อ session จบ")


CKPT_DIR        : /kaggle/working/sied_strong_optimized
RESUME_INPUT_DIR: /kaggle/input/datasets/kmkimmy/sied-strong-optimized
BEST_PARAMS_PATH: /kaggle/working/sied_strong_optimized/best_params.json
FINAL_CKPT_PATH : /kaggle/working/sied_strong_optimized/final_checkpoint.pt
⚠️  อย่าลืม 'Save Version' (commit) เพื่อให้ checkpoint ใน /kaggle/working ไม่หายเมื่อ session จบ


In [7]:
def normalize_text_for_check(text):
    import re
    import unicodedata

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"#\S+", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"[\U00010000-\U0010ffff]", "", text)
    text = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z0-9]+", "", text)
    return text.lower().strip()


df = pd.read_csv(CSV_PATH)

required_columns = [TEXT_COLUMN] + LABEL_COLUMNS

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"dataset ไม่มี column: {col}")

df = df[required_columns].copy()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()

for col in LABEL_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

df["_norm"] = df[TEXT_COLUMN].map(normalize_text_for_check)

print("DATASET PATH:", CSV_PATH)
print("TOTAL ROWS:", len(df))
print("UNIQUE RAW TWEETS:", df[TEXT_COLUMN].nunique())
print("UNIQUE NORMALIZED TWEETS:", df["_norm"].nunique())

print("\nLABEL DISTRIBUTION")
print(df[LABEL_COLUMNS].sum())

print("\nLABEL CARDINALITY")
print(df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())

df = df.drop(columns=["_norm"]).reset_index(drop=True)


DATASET PATH: /kaggle/input/datasets/kmkimmy/sied-thai-final-20000-balanced-multilabel-adjusted/SIED_Thai_FINAL_20000_balanced_multilabel_adjusted.csv
TOTAL ROWS: 20000
UNIQUE RAW TWEETS: 20000
UNIQUE NORMALIZED TWEETS: 20000

LABEL DISTRIBUTION
Happiness    4504
Sadness      4504
Anger        4505
Disgust      4504
Surprise     4504
Fear         4505
dtype: int64

LABEL CARDINALITY
0      106
1    14000
2     4800
3      950
4      144
Name: count, dtype: int64


In [8]:
# ใช้ dataset ใหม่ตัวเดียว split เป็น train / validation / test
# ใช้ stratify_key จาก label combination เพื่อให้ distribution ของ multi-label ใกล้กันขึ้น

def make_stratify_key(frame):
    label_part = frame[LABEL_COLUMNS].astype(str).agg("".join, axis=1)
    card_part = frame[LABEL_COLUMNS].sum(axis=1).astype(str)
    key = card_part + "_" + label_part

    # sklearn stratify ต้องมีอย่างน้อย 2 ตัวต่อ class
    counts = key.value_counts()
    key = key.where(key.map(counts) >= 2, "rare")
    return key


stratify_key = make_stratify_key(df)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=stratify_key
)

temp_stratify_key = make_stratify_key(temp_df)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
    stratify=temp_stratify_key
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

print("\nTRAIN LABEL DISTRIBUTION")
print(train_df[LABEL_COLUMNS].sum())

print("\nVAL LABEL DISTRIBUTION")
print(val_df[LABEL_COLUMNS].sum())

print("\nTEST LABEL DISTRIBUTION")
print(test_df[LABEL_COLUMNS].sum())

print("\nTRAIN LABEL CARDINALITY")
print(train_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())

print("\nVAL LABEL CARDINALITY")
print(val_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())

print("\nTEST LABEL CARDINALITY")
print(test_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())


TRAIN: 14000
VALIDATION: 3000
TEST: 3000

TRAIN LABEL DISTRIBUTION
Happiness    3152
Sadness      3152
Anger        3153
Disgust      3153
Surprise     3152
Fear         3154
dtype: int64

VAL LABEL DISTRIBUTION
Happiness    675
Sadness      675
Anger        676
Disgust      677
Surprise     677
Fear         674
dtype: int64

TEST LABEL DISTRIBUTION
Happiness    677
Sadness      677
Anger        676
Disgust      674
Surprise     675
Fear         677
dtype: int64

TRAIN LABEL CARDINALITY
0      74
1    9801
2    3360
3     665
4     100
Name: count, dtype: int64

VAL LABEL CARDINALITY
0      16
1    2100
2     720
3     142
4      22
Name: count, dtype: int64

TEST LABEL CARDINALITY
0      16
1    2099
2     720
3     143
4      22
Name: count, dtype: int64


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

In [10]:
class EmotionDataset(Dataset):

    def __init__(self, df):

        self.texts = df[TEXT_COLUMN].tolist()

        self.labels = df[LABEL_COLUMNS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        labels = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }
train_dataset = EmotionDataset(train_df)

val_dataset = EmotionDataset(val_df)

test_dataset = EmotionDataset(test_df)

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [12]:
# pos_weight เดิมทำให้ recall สูง แต่ precision ตกได้
# จึงไม่ fix pos_weight ตายตัว แต่ให้ Optuna tune pos_weight_scale
# scale = 0.0 คือไม่ใช้ pos_weight
# scale = 1.0 คือใช้ pos_weight เต็มสูตร negative / positive

label_counts = train_df[LABEL_COLUMNS].sum().values.astype(np.float32)
total_samples = len(train_df)
neg_counts = total_samples - label_counts

pos_weights_raw = neg_counts / (label_counts + 1e-6)

pos_weights_raw = torch.tensor(
    pos_weights_raw,
    dtype=torch.float
).to(device)

def build_pos_weight(pos_weight_scale):
    pos_weight_scale = float(pos_weight_scale)

    weights = 1.0 + (pos_weights_raw - 1.0) * pos_weight_scale

    return weights.to(device)

print("LABEL COUNTS:", label_counts)
print("RAW POS WEIGHTS:", pos_weights_raw)
print("NO POS WEIGHT:", build_pos_weight(0.0))
print("HALF POS WEIGHT:", build_pos_weight(0.5))


LABEL COUNTS: [3152. 3152. 3153. 3153. 3152. 3154.]
RAW POS WEIGHTS: tensor([3.4416, 3.4416, 3.4402, 3.4402, 3.4416, 3.4388], device='cuda:0')
NO POS WEIGHT: tensor([1., 1., 1., 1., 1., 1.], device='cuda:0')
HALF POS WEIGHT: tensor([2.2208, 2.2208, 2.2201, 2.2201, 2.2208, 2.2194], device='cuda:0')


In [13]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)

        weights = torch.softmax(scores, dim=1)

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [14]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.15,
        ff_dropout=0.15,
        ff_mult=2
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        # Fixed Thai encoder: train เฉพาะ decoder + pooling + classifier
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.decoder = Decoder(
            dim=hidden_size,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(hidden_size)

        self.dropout = nn.Dropout(0.30)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(self, input_ids, attention_mask):

        with torch.no_grad():
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        x = outputs.last_hidden_state

        x = self.decoder(x)

        x = self.pooling(x, attention_mask)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits


In [15]:
def optimize_thresholds(
    y_true,
    y_probs,
    min_thr=0.20,
    max_thr=0.80,
    step=0.005
):

    thresholds = np.arange(
        min_thr,
        max_thr + 1e-9,
        step
    )

    best_thresholds = []

    for i in range(NUM_LABELS):

        best_thr = 0.5
        best_score = -1
        best_precision = -1

        true_support = y_true[:, i].sum()

        for thr in thresholds:

            preds = (y_probs[:, i] >= thr).astype(int)

            score = f1_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            precision = precision_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            # ถ้า F1 เท่ากัน ให้เลือก threshold ที่ precision สูงกว่า
            # ช่วยแก้ปัญหา optimized เดิมที่ recall สูงแต่ precision ต่ำ
            if (score > best_score) or (
                np.isclose(score, best_score) and precision > best_precision
            ):

                best_score = score
                best_precision = precision
                best_thr = thr

        best_thresholds.append(best_thr)

    return np.array(best_thresholds)


In [16]:
def evaluate(model, loader, thresholds=None):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask
            )

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    if thresholds is None:
        thresholds = np.array([0.5] * NUM_LABELS)

    preds = (all_probs >= thresholds).astype(int)

    micro_f1 = f1_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        preds,
        average="weighted",
        zero_division=0
    )

    micro_precision = precision_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    micro_recall = recall_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    per_label_f1 = f1_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    per_label_precision = precision_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    per_label_recall = recall_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    subset_accuracy = accuracy_score(
        all_labels,
        preds
    )

    h_loss = hamming_loss(
        all_labels,
        preds
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "micro_precision": micro_precision,
        "macro_precision": macro_precision,
        "micro_recall": micro_recall,
        "macro_recall": macro_recall,
        "per_label_f1": per_label_f1,
        "per_label_precision": per_label_precision,
        "per_label_recall": per_label_recall,
        "min_per_label_f1": float(np.min(per_label_f1)),
        "accuracy": subset_accuracy,
        "subset_accuracy": subset_accuracy,
        "hamming_loss": h_loss,
        "labels": all_labels,
        "preds": preds,
        "probs": all_probs
    }


def objective_score(result):
    # objective ใหม่:
    # - macro_f1 สำคัญที่สุด เพราะต้องชนะราย class
    # - micro_f1 ยังสำคัญ
    # - min_per_label_f1 กันไม่ให้ class ใด class หนึ่งต่ำ โดยเฉพาะ Disgust
    # - subset accuracy ช่วยให้ทายชุด label ตรงขึ้น
    return (
        0.45 * result["macro_f1"]
        + 0.25 * result["micro_f1"]
        + 0.20 * result["min_per_label_f1"]
        + 0.10 * result["subset_accuracy"]
    )


In [17]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay,
    epochs=None,
    pos_weight_scale=0.0,
    label_smoothing=0.0,
    loss_name="bce",
    focal_gamma=2.0,
    verbose=True,
    ckpt_path=None,        # WRITE: ที่เซฟ checkpoint (เปิด checkpoint/resume เมื่อระบุ)
    resume=False,          # True = โหลด checkpoint แล้วเทรนต่อ
    resume_path=None,      # READ: path ที่ใช้โหลด (เช่นจาก /kaggle/input) ถ้า None ใช้ ckpt_path
    checkpoint_every=5     # save checkpoint ทุกกี่ epoch
):

    if epochs is None:
        epochs = EPOCHS

    pos_weight = build_pos_weight(pos_weight_scale)

    bce_loss = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight,
        reduction="none"
    )

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = len(train_loader) * epochs

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.06 * total_steps),
        num_training_steps=total_steps
    )

    use_amp = device.type == "cuda"

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    # ===== RESUME: โหลด checkpoint ถ้ามี =====
    # หมายเหตุ: scheduler ผูกกับ total_steps (= len(train_loader) * epochs)
    #          ดังนั้นตอน resume ต้องใช้ epochs เท่าเดิม (EPOCHS) ไม่งั้น schedule จะเพี้ยน
    start_epoch = 0
    load_from = None
    if resume:
        if resume_path is not None and os.path.exists(resume_path):
            load_from = resume_path
        elif ckpt_path is not None and os.path.exists(ckpt_path):
            load_from = ckpt_path

    if load_from is not None:
        ckpt = torch.load(load_from, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"]   # epoch ถัดไปที่ต้องเริ่ม
        print(f"✅ RESUMED from checkpoint: {load_from}")
        print(f"   เริ่มต่อจาก epoch {start_epoch + 1}/{epochs}")
        if start_epoch >= epochs:
            print("   ✔️ เทรนครบแล้ว ไม่ต้องเทรนต่อ")
            return model
    elif ckpt_path is not None:
        print(f"ℹ️  ไม่พบ checkpoint -> เริ่มเทรนใหม่จาก epoch 1 (จะเซฟไป {ckpt_path})")

    def save_checkpoint(next_epoch):
        if ckpt_path is None:
            return
        # save ลง temp ก่อนแล้ว replace เพื่อกัน checkpoint พังถ้า kernel ตายระหว่าง save
        tmp_path = ckpt_path + ".tmp"
        torch.save({
            "epoch": next_epoch,                                # epoch ถัดไปที่ต้องเริ่ม
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }, tmp_path)
        os.replace(tmp_path, ckpt_path)

    for epoch in range(start_epoch, epochs):

        model.train()
        total_loss = 0

        loop = tqdm(
            train_loader,
            disable=not verbose
        )

        for batch in loop:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            if label_smoothing > 0:
                # smooth toward 0.5 for multi-label BCE
                labels_for_loss = labels * (1.0 - label_smoothing) + 0.5 * label_smoothing
            else:
                labels_for_loss = labels

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss_matrix = bce_loss(
                    logits,
                    labels_for_loss
                )

                if loss_name == "focal":
                    probs = torch.sigmoid(logits)
                    pt = probs * labels + (1 - probs) * (1 - labels)
                    focal_weight = (1 - pt).pow(focal_gamma)
                    loss = (focal_weight * loss_matrix).mean()
                else:
                    loss = loss_matrix.mean()

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

            if verbose:
                loop.set_description(
                    f"Epoch {epoch+1}/{epochs}"
                )

                loop.set_postfix(
                    loss=loss.item()
                )

        # ===== SAVE CHECKPOINT ทุก N epoch + epoch สุดท้าย =====
        if ckpt_path is not None and (
            (epoch + 1) % checkpoint_every == 0 or (epoch + 1) == epochs
        ):
            save_checkpoint(epoch + 1)
            if verbose:
                print(f"  💾 Checkpoint saved (เทรนเสร็จ epoch {epoch+1}) -> {ckpt_path}")

        if verbose:

            val_raw = evaluate(
                model,
                val_loader
            )

            val_thresholds = optimize_thresholds(
                val_raw["labels"],
                val_raw["probs"]
            )

            val_result = evaluate(
                model,
                val_loader,
                val_thresholds
            )

            print("\n======================")
            print(f"Epoch {epoch+1}")
            print("======================")

            print(
                f"VAL SUBSET ACC : {val_result['subset_accuracy']:.4f}"
            )

            print(
                f"VAL MICRO P/R/F1: "
                f"{val_result['micro_precision']:.4f} / "
                f"{val_result['micro_recall']:.4f} / "
                f"{val_result['micro_f1']:.4f}"
            )

            print(
                f"VAL MACRO P/R/F1: "
                f"{val_result['macro_precision']:.4f} / "
                f"{val_result['macro_recall']:.4f} / "
                f"{val_result['macro_f1']:.4f}"
            )

            print(
                f"VAL MIN LABEL F1: {val_result['min_per_label_f1']:.4f}"
            )

            print(
                f"VAL WEIGHTED F1: {val_result['weighted_f1']:.4f}"
            )

            print(
                f"VAL HAMMING LOSS: {val_result['hamming_loss']:.4f}"
            )

    return model


In [18]:
def objective(trial):

    # baseline trial จะถูก enqueue ไว้ก่อนหน้า
    depth = trial.suggest_int(
        "depth",
        1,
        3
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4, 8]
    )

    # ลด dropout range จากเดิมที่สูงเกินไป เพราะเดิม optimized มีแนวโน้ม underfit / recall เยอะ precision ตก
    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.05,
        0.30
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.05,
        0.30
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        1e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        5e-3,
        log=True
    )

    # จุดสำคัญ: ให้ Optuna เลือกว่าจะใช้ pos_weight แค่ไหน
    # ถ้า scale สูงเกินจะ recall สูงแต่ precision ลด
    pos_weight_scale = trial.suggest_float(
        "pos_weight_scale",
        0.0,
        0.60
    )

    label_smoothing = trial.suggest_float(
        "label_smoothing",
        0.0,
        0.04
    )

    loss_name = trial.suggest_categorical(
        "loss_name",
        ["bce", "focal"]
    )

    focal_gamma = trial.suggest_float(
        "focal_gamma",
        1.0,
        2.5
    )

    model = EmotionModel(
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(device)

    # หมายเหตุ: ตอน Optuna search ไม่ต้องทำ checkpoint (แต่ละ trial เป็นคนละ model)
    model = train_model(
        model,
        train_loader,
        val_loader,
        lr=lr,
        weight_decay=weight_decay,
        epochs=OPTUNA_EPOCHS,
        pos_weight_scale=pos_weight_scale,
        label_smoothing=label_smoothing,
        loss_name=loss_name,
        focal_gamma=focal_gamma,
        verbose=False
    )

    val_raw = evaluate(
        model,
        val_loader
    )

    thresholds = optimize_thresholds(
        val_raw["labels"],
        val_raw["probs"]
    )

    val_result = evaluate(
        model,
        val_loader,
        thresholds
    )

    score = objective_score(val_result)

    trial.set_user_attr("val_macro_f1", float(val_result["macro_f1"]))
    trial.set_user_attr("val_micro_f1", float(val_result["micro_f1"]))
    trial.set_user_attr("val_min_per_label_f1", float(val_result["min_per_label_f1"]))
    trial.set_user_attr("val_subset_accuracy", float(val_result["subset_accuracy"]))
    trial.set_user_attr("thresholds", thresholds.tolist())

    return score


In [19]:
# ===== OPTUNA SEARCH (ข้ามได้ถ้าเคย search แล้ว) =====
# ถ้ามี best_params.json อยู่แล้ว (ใน /kaggle/working หรือ dataset ที่ attach) -> โหลดมาใช้เลย
# สำคัญมากตอน resume: session หลุดทำให้ study หาย ถ้าไม่ save best_params ก็ต้อง search 25 trials ใหม่
_bp_path = resolve_resume("best_params.json", BEST_PARAMS_PATH)

if _bp_path is not None:
    with open(_bp_path, "r") as f:
        best_params = json.load(f)
    study = None
    print(f"✅ พบ best_params เดิม ({_bp_path}) -> ข้าม Optuna search")
    print("best_params:", best_params)
    # copy มาไว้ใน /kaggle/working ด้วย เผื่อจะ Save Version รวมไว้ที่เดียว
    if _bp_path != BEST_PARAMS_PATH:
        with open(BEST_PARAMS_PATH, "w") as f:
            json.dump(best_params, f, ensure_ascii=False, indent=2)
else:
    sampler = optuna.samplers.TPESampler(
        seed=SEED,
        multivariate=True
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler
    )

    # ใส่ค่า baseline/non-optimized configuration เป็น trial แรก
    # เพื่อให้ optimization มีจุดเริ่มต้นที่ไม่แพ้ baseline ง่าย ๆ
    study.enqueue_trial({
        "depth": 1,
        "heads": 2,
        "attn_dropout": 0.20,
        "ff_dropout": 0.20,
        "ff_mult": 2,
        "lr": 2e-5,
        "weight_decay": 1e-2,
        "pos_weight_scale": 0.0,
        "label_smoothing": 0.0,
        "loss_name": "bce",
        "focal_gamma": 2.0
    })

    study.optimize(
        objective,
        n_trials=N_TRIALS
    )

    # save best_params ทันทีที่ search เสร็จ -> resume ครั้งหน้าไม่ต้อง search ใหม่
    best_params = study.best_params
    with open(BEST_PARAMS_PATH, "w") as f:
        json.dump(best_params, f, ensure_ascii=False, indent=2)
    print("💾 SAVED best_params ->", BEST_PARAMS_PATH)


✅ พบ best_params เดิม (/kaggle/input/datasets/kmkimmy/sied-strong-optimized/best_params.json) -> ข้าม Optuna search
best_params: {'depth': 3, 'heads': 4, 'attn_dropout': 0.16017924409392248, 'ff_dropout': 0.27061369851865547, 'ff_mult': 3, 'lr': 9.293634078185081e-05, 'weight_decay': 9.167446093309075e-06, 'pos_weight_scale': 0.27649809292042143, 'label_smoothing': 0.014745457985929526, 'loss_name': 'bce', 'focal_gamma': 2.0644269381863416}


In [20]:
# study เป็น None ตอน resume (โหลด best_params จากไฟล์ ไม่ได้ search ใหม่)
if study is not None:
    print("\n======================")
    print("BEST PARAMETERS")
    print("======================")

    print(study.best_params)

    print("\n======================")
    print("BEST TRIAL VALIDATION ATTRS")
    print("======================")

    print("objective_score:", study.best_value)
    print("val_macro_f1:", study.best_trial.user_attrs.get("val_macro_f1"))
    print("val_micro_f1:", study.best_trial.user_attrs.get("val_micro_f1"))
    print("val_min_per_label_f1:", study.best_trial.user_attrs.get("val_min_per_label_f1"))
    print("val_subset_accuracy:", study.best_trial.user_attrs.get("val_subset_accuracy"))
else:
    print("ℹ️  ข้าม Optuna (resume) — ใช้ best_params จากไฟล์:")
    print(best_params)


ℹ️  ข้าม Optuna (resume) — ใช้ best_params จากไฟล์:
{'depth': 3, 'heads': 4, 'attn_dropout': 0.16017924409392248, 'ff_dropout': 0.27061369851865547, 'ff_mult': 3, 'lr': 9.293634078185081e-05, 'weight_decay': 9.167446093309075e-06, 'pos_weight_scale': 0.27649809292042143, 'label_smoothing': 0.014745457985929526, 'loss_name': 'bce', 'focal_gamma': 2.0644269381863416}


In [21]:
# สร้าง final_model ด้วย best_params (โครงสร้างต้องตรงกับ checkpoint ตอน resume)
final_model = EmotionModel(
    depth=best_params["depth"],
    heads=best_params["heads"],
    attn_dropout=best_params["attn_dropout"],
    ff_dropout=best_params["ff_dropout"],
    ff_mult=best_params["ff_mult"]
).to(device)


model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

In [22]:
# ===== FINAL TRAINING (มี checkpoint + resume กัน Kaggle session หลุด) =====
# ถ้า session หลุดแล้วรันใหม่ทั้งหมด: cell นี้จะโหลด checkpoint แล้วเทรนต่อจาก epoch ล่าสุด
#   - ภายใน session เดียว: โหลดจาก /kaggle/working
#   - ข้าม session: ตั้ง RESUME_INPUT_DIR (cell config) ให้ชี้ dataset ที่ attach -> โหลดจากที่นั่น
# best_params ถูกข้าม Optuna ไปแล้ว -> final_model สร้างด้วย params เดิม โครงสร้างตรงกับ checkpoint แน่นอน
final_model = train_model(
    final_model,
    train_loader,
    val_loader,
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"],
    epochs=EPOCHS,
    pos_weight_scale=best_params["pos_weight_scale"],
    label_smoothing=best_params["label_smoothing"],
    loss_name=best_params["loss_name"],
    focal_gamma=best_params["focal_gamma"],
    verbose=True,
    ckpt_path=FINAL_CKPT_PATH,                                          # WRITE -> /kaggle/working
    resume=True,                                                        # โหลด checkpoint ถ้ามี
    resume_path=resolve_resume("final_checkpoint.pt", FINAL_CKPT_PATH), # READ (working ก่อน แล้ว input dataset)
    checkpoint_every=CHECKPOINT_EVERY
)


✅ RESUMED from checkpoint: /kaggle/input/datasets/kmkimmy/sied-strong-optimized/final_checkpoint.pt
   เริ่มต่อจาก epoch 16/100


Epoch 16/100: 100%|██████████| 219/219 [00:45<00:00,  4.84it/s, loss=0.328]



Epoch 16
VAL SUBSET ACC : 0.6820
VAL MICRO P/R/F1: 0.8996 / 0.7760 / 0.8333
VAL MACRO P/R/F1: 0.8999 / 0.7760 / 0.8332
VAL MIN LABEL F1: 0.8063
VAL WEIGHTED F1: 0.8332
VAL HAMMING LOSS: 0.0699


Epoch 17/100: 100%|██████████| 219/219 [00:46<00:00,  4.71it/s, loss=0.309]



Epoch 17
VAL SUBSET ACC : 0.6850
VAL MICRO P/R/F1: 0.9096 / 0.7699 / 0.8339
VAL MACRO P/R/F1: 0.9100 / 0.7699 / 0.8340
VAL MIN LABEL F1: 0.8061
VAL WEIGHTED F1: 0.8340
VAL HAMMING LOSS: 0.0691


...


Epoch 98
VAL SUBSET ACC : 0.6973
VAL MICRO P/R/F1: 0.9275 / 0.7696 / 0.8412
VAL MACRO P/R/F1: 0.9280 / 0.7696 / 0.8412
VAL MIN LABEL F1: 0.8127
VAL WEIGHTED F1: 0.8412
VAL HAMMING LOSS: 0.0654


Epoch 99/100: 100%|██████████| 219/219 [00:50<00:00,  4.32it/s, loss=0.266]



Epoch 99
VAL SUBSET ACC : 0.6970
VAL MICRO P/R/F1: 0.9274 / 0.7694 / 0.8410
VAL MACRO P/R/F1: 0.9280 / 0.7694 / 0.8411
VAL MIN LABEL F1: 0.8127
VAL WEIGHTED F1: 0.8410
VAL HAMMING LOSS: 0.0655


Epoch 100/100: 100%|██████████| 219/219 [00:50<00:00,  4.32it/s, loss=0.258]


  💾 Checkpoint saved (เทรนเสร็จ epoch 100) -> /kaggle/working/sied_strong_optimized/final_checkpoint.pt

Epoch 100
VAL SUBSET ACC : 0.6973
VAL MICRO P/R/F1: 0.9274 / 0.7694 / 0.8410
VAL MACRO P/R/F1: 0.9280 / 0.7694 / 0.8411
VAL MIN LABEL F1: 0.8127
VAL WEIGHTED F1: 0.8410
VAL HAMMING LOSS: 0.0655


In [23]:
val_result = evaluate(
    final_model,
    val_loader
)

best_thresholds = optimize_thresholds(
    val_result["labels"],
    val_result["probs"]
)

val_result_tuned = evaluate(
    final_model,
    val_loader,
    best_thresholds
)

print("\n======================")
print("BEST THRESHOLDS FROM FINAL VALIDATION")
print("======================")

for label, thr in zip(
    LABEL_COLUMNS,
    best_thresholds
):
    print(f"{label}: {thr:.3f}")

print("\nVALIDATION RESULT AFTER THRESHOLD TUNING")
print("macro_f1:", val_result_tuned["macro_f1"])
print("micro_f1:", val_result_tuned["micro_f1"])
print("min_per_label_f1:", val_result_tuned["min_per_label_f1"])
print("subset_accuracy:", val_result_tuned["subset_accuracy"])



BEST THRESHOLDS FROM FINAL VALIDATION
Happiness: 0.710
Sadness: 0.570
Anger: 0.680
Disgust: 0.710
Surprise: 0.630
Fear: 0.715

VALIDATION RESULT AFTER THRESHOLD TUNING
macro_f1: 0.8410658302442027
micro_f1: 0.841040852096535
min_per_label_f1: 0.8127035830618893
subset_accuracy: 0.6973333333333334


In [24]:
def summarize_result(name, result):

    return {
        "split": name,
        "subset_accuracy_exact_match": result["subset_accuracy"],
        "micro_precision": result["micro_precision"],
        "micro_recall": result["micro_recall"],
        "micro_f1": result["micro_f1"],
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "macro_f1": result["macro_f1"],
        "weighted_f1": result["weighted_f1"],
        "min_per_label_f1": result["min_per_label_f1"],
        "hamming_loss": result["hamming_loss"]
    }


train_result = evaluate(
    final_model,
    train_eval_loader,
    best_thresholds
)

val_result_final = evaluate(
    final_model,
    val_loader,
    best_thresholds
)

test_result = evaluate(
    final_model,
    test_loader,
    best_thresholds
)

metrics_summary = pd.DataFrame([
    summarize_result("train", train_result),
    summarize_result("validation", val_result_final),
    summarize_result("test", test_result)
])

print("\n======================")
print("FINAL METRICS SUMMARY")
print("======================")
display(metrics_summary)

print("\n======================")
print("BEST THRESHOLDS")
print("======================")
threshold_df = pd.DataFrame({
    "label": LABEL_COLUMNS,
    "threshold": best_thresholds
})
display(threshold_df)

print("\n======================")
print("TEST SET MAIN RESULT")
print("======================")

print(
    f"Subset Accuracy / Exact Match : {test_result['subset_accuracy']:.4f}"
)

print(
    f"Micro Precision              : {test_result['micro_precision']:.4f}"
)

print(
    f"Micro Recall                 : {test_result['micro_recall']:.4f}"
)

print(
    f"Micro F1                     : {test_result['micro_f1']:.4f}"
)

print(
    f"Macro Precision              : {test_result['macro_precision']:.4f}"
)

print(
    f"Macro Recall                 : {test_result['macro_recall']:.4f}"
)

print(
    f"Macro F1                     : {test_result['macro_f1']:.4f}"
)

print(
    f"Weighted F1                  : {test_result['weighted_f1']:.4f}"
)

print(
    f"Min Per-label F1             : {test_result['min_per_label_f1']:.4f}"
)

print(
    f"Hamming Loss                 : {test_result['hamming_loss']:.4f}"
)

print("\n======================")
print("COMPARE WITH NON-OPTIMIZED REFERENCE")
print("======================")

reference_macro = NON_OPTIMIZED_REFERENCE["macro_f1_estimated_from_per_label"]

print("Non-optimized macro F1 reference:", reference_macro)
print("Current optimized macro F1:", test_result["macro_f1"])
print("Difference:", test_result["macro_f1"] - reference_macro)



FINAL METRICS SUMMARY


,split,subset_accuracy_exact_match,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_f1,min_per_label_f1,hamming_loss
0,train,0.719357,0.948956,0.790178,0.862319,0.949525,0.790181,0.862189,0.862188,0.835779,0.056821
1,validation,0.697333,0.927446,0.769364,0.841041,0.927988,0.769379,0.841066,0.841048,0.812704,0.065500
2,test,0.694333,0.928294,0.766026,0.839389,0.928710,0.766015,0.839212,0.839220,0.823045,0.066056



BEST THRESHOLDS


,label,threshold
0,Happiness,0.710
1,Sadness,0.570
2,Anger,0.680
3,Disgust,0.710
4,Surprise,0.630
5,Fear,0.715



TEST SET MAIN RESULT
Subset Accuracy / Exact Match : 0.6943
Micro Precision              : 0.9283
Micro Recall                 : 0.7660
Micro F1                     : 0.8394
Macro Precision              : 0.9287
Macro Recall                 : 0.7660
Macro F1                     : 0.8392
Weighted F1                  : 0.8392
Min Per-label F1             : 0.8230
Hamming Loss                 : 0.0661

COMPARE WITH NON-OPTIMIZED REFERENCE
Non-optimized macro F1 reference: 0.8234
Current optimized macro F1: 0.8392117608592318
Difference: 0.01581176085923175


In [25]:
print("\n======================")
print("TEST CLASSIFICATION REPORT")
print("======================")

report_dict = classification_report(
    test_result["labels"],
    test_result["preds"],
    target_names=LABEL_COLUMNS,
    zero_division=0,
    output_dict=True
)

per_label_report = pd.DataFrame(report_dict).T

display(per_label_report)

per_label_compare = pd.DataFrame({
    "label": LABEL_COLUMNS,
    "precision": test_result["per_label_precision"],
    "recall": test_result["per_label_recall"],
    "f1_score": test_result["per_label_f1"],
    "non_optimized_reference_f1": [
        NON_OPTIMIZED_REFERENCE["happiness_f1"],
        NON_OPTIMIZED_REFERENCE["sadness_f1"],
        NON_OPTIMIZED_REFERENCE["anger_f1"],
        NON_OPTIMIZED_REFERENCE["disgust_f1"],
        NON_OPTIMIZED_REFERENCE["surprise_f1"],
        NON_OPTIMIZED_REFERENCE["fear_f1"],
    ]
})

per_label_compare["f1_difference"] = (
    per_label_compare["f1_score"]
    - per_label_compare["non_optimized_reference_f1"]
)

print("\n======================")
print("PER-LABEL COMPARISON WITH NON-OPTIMIZED REFERENCE")
print("======================")

display(per_label_compare)

print("\n======================")
print("TEST MULTI-LABEL CONFUSION MATRIX")
print("======================")

mcm = multilabel_confusion_matrix(
    test_result["labels"],
    test_result["preds"]
)

confusion_rows = []

for i, label in enumerate(LABEL_COLUMNS):
    tn, fp, fn, tp = mcm[i].ravel()

    confusion_rows.append({
        "label": label,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn)
    })

confusion_df = pd.DataFrame(confusion_rows)

display(confusion_df)

print("\n======================")
print("TEST LABEL SUPPORT / PREDICTION COUNTS")
print("======================")

support_pred_df = pd.DataFrame({
    "label": LABEL_COLUMNS,
    "true_support": test_result["labels"].sum(axis=0).astype(int),
    "predicted_positive": test_result["preds"].sum(axis=0).astype(int),
    "threshold": best_thresholds
})

display(support_pred_df)

test_probs_df = pd.DataFrame(
    test_result["probs"],
    columns=[f"prob_{label}" for label in LABEL_COLUMNS]
)

test_preds_df = pd.DataFrame(
    test_result["preds"],
    columns=[f"pred_{label}" for label in LABEL_COLUMNS]
)

test_true_df = test_df[[TEXT_COLUMN] + LABEL_COLUMNS].copy()

test_predictions_output = pd.concat(
    [
        test_true_df.reset_index(drop=True),
        test_probs_df.reset_index(drop=True),
        test_preds_df.reset_index(drop=True)
    ],
    axis=1
)

# เขียน output ลง CKPT_DIR (/kaggle/working) — จะถูกเก็บเมื่อ Save Version
export_targets = {
    "sied_strong_optimized_metrics_summary.csv": metrics_summary,
    "sied_strong_optimized_thresholds.csv": threshold_df,
    "sied_strong_optimized_per_label_compare.csv": per_label_compare,
    "sied_strong_optimized_confusion_matrix.csv": confusion_df,
    "sied_strong_optimized_support_prediction_counts.csv": support_pred_df,
    "sied_strong_optimized_test_predictions.csv": test_predictions_output,
}

print("\nEXPORTED FILES")
for fname, frame in export_targets.items():
    out_path = os.path.join(CKPT_DIR, fname)
    frame.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(out_path)

# per_label_report เซฟแยก (มี index เป็นชื่อ label)
per_label_report_path = os.path.join(CKPT_DIR, "sied_strong_optimized_per_label_report.csv")
per_label_report.to_csv(per_label_report_path, encoding="utf-8-sig")
print(per_label_report_path)



TEST CLASSIFICATION REPORT


,precision,recall,f1-score,support
Happiness,0.952030,0.762186,0.846596,677.0
Sadness,0.908638,0.807976,0.855356,677.0
Anger,0.920863,0.757396,0.831169,676.0
Disgust,0.924214,0.741840,0.823045,674.0
Surprise,0.928196,0.785185,0.850722,675.0
Fear,0.938318,0.741507,0.828383,677.0
micro avg,0.928294,0.766026,0.839389,4056.0
macro avg,0.928710,0.766015,0.839212,4056.0
weighted avg,0.928715,0.766026,0.839220,4056.0
samples avg,0.943206,0.839833,0.871267,4056.0



PER-LABEL COMPARISON WITH NON-OPTIMIZED REFERENCE


,label,precision,recall,f1_score,non_optimized_reference_f1,f1_difference
0,Happiness,0.952030,0.762186,0.846596,0.8398,0.006796
1,Sadness,0.908638,0.807976,0.855356,0.8334,0.021956
2,Anger,0.920863,0.757396,0.831169,0.8131,0.018069
3,Disgust,0.924214,0.741840,0.823045,0.7892,0.033845
4,Surprise,0.928196,0.785185,0.850722,0.8310,0.019722
5,Fear,0.938318,0.741507,0.828383,0.8341,-0.005717



TEST MULTI-LABEL CONFUSION MATRIX


,label,TP,FP,FN,TN
0,Happiness,516,26,161,2297
1,Sadness,547,55,130,2268
2,Anger,512,44,164,2280
3,Disgust,500,41,174,2285
4,Surprise,530,41,145,2284
5,Fear,502,33,175,2290



TEST LABEL SUPPORT / PREDICTION COUNTS


,label,true_support,predicted_positive,threshold
0,Happiness,677,542,0.710
1,Sadness,677,602,0.570
2,Anger,676,556,0.680
3,Disgust,674,541,0.710
4,Surprise,675,571,0.630
5,Fear,677,535,0.715



EXPORTED FILES
/kaggle/working/sied_strong_optimized/sied_strong_optimized_metrics_summary.csv
/kaggle/working/sied_strong_optimized/sied_strong_optimized_thresholds.csv
/kaggle/working/sied_strong_optimized/sied_strong_optimized_per_label_compare.csv
/kaggle/working/sied_strong_optimized/sied_strong_optimized_confusion_matrix.csv
/kaggle/working/sied_strong_optimized/sied_strong_optimized_support_prediction_counts.csv
/kaggle/working/sied_strong_optimized/sied_strong_optimized_test_predictions.csv
/kaggle/working/sied_strong_optimized/sied_strong_optimized_per_label_report.csv


In [26]:
# เซฟ final model ลง /kaggle/working (เป็น dict: state_dict + thresholds + best_params + labels)
# ⚠️ ตอนโหลดไปใช้ inference ต้องใช้ thresholds ที่เซฟไว้ ไม่ใช่ 0.5
SAVE_PATH = os.path.join(CKPT_DIR, "final_sied_wangchanberta_xtransformers_strong_optimized.pt")

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "thresholds": best_thresholds,
        "best_params": best_params,
        "labels": LABEL_COLUMNS,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "dataset": "SIED_Thai_FINAL_20000_balanced_multilabel_adjusted.csv",
        "split": "single new dataset; stratified train 70%, validation 15%, test 15%",
        "objective": "0.45 macro_f1 + 0.25 micro_f1 + 0.20 min_per_label_f1 + 0.10 subset_accuracy",
        "note": "Strong optimized version; encoder frozen; pos_weight_scale and thresholds tuned; final model saved after 100 epochs."
    },
    SAVE_PATH
)

print("\nMODEL SAVED:", SAVE_PATH)
print("💡 อย่าลืม 'Save Version' (commit) เพื่อเก็บไฟล์ทั้งหมดใน /kaggle/working")



MODEL SAVED: /kaggle/working/sied_strong_optimized/final_sied_wangchanberta_xtransformers_strong_optimized.pt
💡 อย่าลืม 'Save Version' (commit) เพื่อเก็บไฟล์ทั้งหมดใน /kaggle/working
